In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pymargins import Margins

rng = np.random.default_rng(42)
n = 2000
df = pd.DataFrame({
    "age": rng.integers(20, 75, n),
    "female": rng.binomial(1, 0.52, n),
    "treated": rng.binomial(1, 0.40, n),
})
lp = -1.5 + 0.04 * df["age"] - 0.3 * df["female"] + 0.8 * df["treated"]
df["y"] = rng.binomial(1, 1 / (1 + np.exp(-lp)))

fit = smf.glm("y ~ age + female + treated", data=df,
              family=sm.families.Binomial()).fit()
m = Margins.log_scale(fit, at="overall")

In [2]:
m = Margins.log_scale(fit, method="bootstrap", n_boot=2000, vcov="HC3")
print(m.dydx("age").summary())

          Margins Result (bootstrap, level=0.95)          
     estimate  std err  statistic  P>|z|  [95% Conf. Int.]
----------------------------------------------------------
age    0.0088   0.0633    -4.7302  0.000    0.0077, 0.0098

n = 2000
Note: std err is on the inference scale; estimate and CI are on the reporting scale.
κ: 0.117


In [3]:
m = Margins.log_scale(fit, method="bootstrap", n_boot=2000, n_jobs=-1)